<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2: *Data Merging*
##### Version Number: 4.0
---
### Contents  
> *Merge Weather Data*\
> *Merge NDVI Index Data* (Appendix F)\
> *Export File*
---
### Notes
- Integrate sampling grid data with daily weather data from gridMET and NDVI index. Sampling grid includes topological, social, inrastructure, and land cover data.
### Inputs
- Daily Weather Readings - `gridmet_final.csv` 
- NDVI Index Data - `NDVI_index.csv` (built in Appendix F)
- California Mesh Sampling Grid - `Sampling_grids.csv` (built in appendix A) 
---
### Outputs  
- `samples_weather_NDVI.csv` Sampling grid dataset merged with weather data and NDVI index.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

### Data Loading

In [3]:
weather = pd.read_csv('../data/raw/gridmet/gridmet_final.csv')
samples = pd.read_csv("../data/processed/cleaned_grids.csv")
NDVI_index = pd.read_csv('../data/processed/NDVI_index.csv')

#### Add geometry to sampling grid

In [4]:
samples['geometry'] = [Point(xy) for xy in zip(samples['centroid_easting'], samples['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(samples, geometry='geometry', crs="EPSG:3310")

#### Filter out unused weather stations
- some station data in the weather data set are unreferenced by the sampling grid. Since merging is onto the weather data, these will produce many NA values. Only keep those stations that are in the sampling grid.

In [5]:
sample_ids = samples['Sample_ID'].unique()
newweather = weather[weather['Sample_ID'].isin(sample_ids)]

#### Merge on Sample_ID

- Weather data was extracted with different sampling points and extraction is a lengthy process on limited hardware. This merge joins the old points with the closest centorid in the sampling grid. 

In [6]:
# Merge on BOTH station and date
samples_weather = newweather.merge(
    samples_gdf,
    on=['Sample_ID'],
    how='left'
)

In [7]:
post_merge_check (samples_weather, newweather)

Premerged shape:  (425700, 22)
Merged shape:  (608880, 63)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


### Merge NDVI index data

In [8]:
# Merge on BOTH station and date
samples_weather_NDVI = samples_weather.merge(
    NDVI_index,
    on=['grid_id','Date'],
    how='left'
)

In [9]:
post_merge_check (samples_weather_NDVI, samples_weather)

Premerged shape:  (608880, 63)
Merged shape:  (608880, 65)


Duplicates before merge:  0


Duplicates after merge:  0


NA values before merge:  0


NA values after merge:  0


---

## 3. Export File

In [10]:
samples_weather_NDVI.to_csv('../data/processed/samples_weather_NDVI.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
